# 07 — `None` and Truthiness

## Learning objectives

By the end of this notebook, you should be able to:

- Explain what `None` means in Python.
- Recognize the difference between `None`, `False`, `0`, `""`, and an empty collection.
- Check for `None` correctly with `is` and `is not`.
- Explain truthy and falsy values.
- Use truthiness safely in conditions and default-value expressions.
- Avoid common bugs caused by treating “missing” and “empty” as the same thing.

**How to use this notebook:** Run the code cells from top to bottom. Read the output before moving to the next section. Exercises and their solutions are near the end.

## 1. What is `None`?

`None` is Python's special object that represents the absence of a value. It is often used when a value is not available, not yet known, or intentionally empty.

For example, a function that only prints something and does not explicitly return a value returns `None`.

In [1]:
result = print("Hello, Python!")
print("The return value is:", result)
print("Its type is:", type(result))

Hello, Python!
The return value is: None
Its type is: <class 'NoneType'>


`None` is different from the number `0`, the Boolean `False`, and an empty string `""`. Those are real values with their own meanings.

In [2]:
no_value = None
zero = 0
false_value = False
empty_text = ""
empty_list = []

print(no_value)
print(zero)
print(false_value)
print(repr(empty_text))
print(empty_list)

None
0
False
''
[]


## 2. The type of `None`

The type of `None` is `NoneType`. In ordinary Python code, you usually check whether a value is `None` directly rather than writing `type(value) == type(None)`.

In [3]:
value = None

print(type(value))
print(value is None)
print(value is not None)

<class 'NoneType'>
True
False


## 3. Check `None` with `is` and `is not`

Use `is None` to check whether a value is the actual `None` object. Use `is not None` to check that it is not.

Do **not** use `== None` as your normal style. Equality (`==`) asks whether values compare equal; identity (`is`) asks whether two references point to the same object. `None` checks are an important case where identity is the intended test.

In [4]:
username = None

if username is None:
    print("No username has been provided.")
else:
    print("Username:", username)

username = "maya"

if username is not None:
    print("A username was provided:", username)

No username has been provided.
A username was provided: maya


### A subtle but important distinction

A value can be present even when it is falsy. For example, `0` may be a valid score, and `""` may be a valid (though empty) text value. Use an explicit `is None` check when you specifically care whether a value is missing.

In [5]:
score = 0

if score is None:
    print("Score is missing.")
else:
    print("Score was provided:", score)

optional_note = ""
if optional_note is None:
    print("No note value was supplied.")
else:
    print("A note value was supplied; its text is:", repr(optional_note))

Score was provided: 0
A note value was supplied; its text is: ''


## 4. What is truthiness?

Python lets many objects be tested in conditions such as `if value:`. Python evaluates each object as either truthy or falsy.

Common falsy values include:

- `None`
- `False`
- Numeric zero, such as `0`, `0.0`, and `0j`
- Empty strings, such as `""`
- Empty collections, such as `[]`, `()`, `{}`, and `set()`

Most other values are truthy, including non-empty strings and collections. Truthiness is not the same thing as being `True`: a truthy object does not have to be the Boolean object `True`.

In [6]:
examples = [
    ("None", None),
    ("False", False),
    ("0", 0),
    ("0.0", 0.0),
    ("0j", 0j),
    ("empty string", ""),
    ("empty list", []),
    ("empty tuple", ()),
    ("empty dictionary", {}),
    ("empty set", set()),
    ("non-empty string", "hello"),
    ("negative number", -3),
    ("non-empty list", [0]),
]

for label, value in examples:
    print(f"{label:20} -> bool(value) = {bool(value)}")

None                 -> bool(value) = False
False                -> bool(value) = False
0                    -> bool(value) = False
0.0                  -> bool(value) = False
0j                   -> bool(value) = False
empty string         -> bool(value) = False
empty list           -> bool(value) = False
empty tuple          -> bool(value) = False
empty dictionary     -> bool(value) = False
empty set            -> bool(value) = False
non-empty string     -> bool(value) = True
negative number      -> bool(value) = True
non-empty list       -> bool(value) = True


## 5. `bool()` and conditions

`bool(value)` shows the truth value Python uses in a condition. You normally do not need to call `bool()` inside an `if`; `if value:` already performs the truth test.

In [7]:
items = []

print("Explicit conversion:", bool(items))

if items:
    print("There are items.")
else:
    print("The list is empty.")

Explicit conversion: False
The list is empty.


### Truthiness does not change the original value

Testing a value in a condition does not convert the variable into `True` or `False`. The original object remains unchanged.

In [8]:
value = "Python"

if value:
    print("The value is truthy.")

print("Original value:", value)
print("Original type:", type(value))

The value is truthy.
Original value: Python
Original type: <class 'str'>


## 6. `None` versus falsy values

Consider these two conditions:

- `if value is None:` checks specifically for missing/`None`.
- `if not value:` checks whether the value is falsy, which includes `None`, `False`, `0`, `""`, and empty collections.

They answer different questions.

In [9]:
def describe_value(value):
    if value is None:
        return "The value is None."
    if not value:
        return "The value exists, but it is falsy."
    return "The value is truthy."

test_values = [None, False, 0, "", [], "hello", [1]]

for item in test_values:
    print(f"{item!r:10} -> {describe_value(item)}")

None       -> The value is None.
False      -> The value exists, but it is falsy.
0          -> The value exists, but it is falsy.
''         -> The value exists, but it is falsy.
[]         -> The value exists, but it is falsy.
'hello'    -> The value is truthy.
[1]        -> The value is truthy.


## 7. Common use: optional function arguments

A function may accept an optional value. If the caller passes `None`, the function can decide what default to use.

Be careful: if you use `if not value`, you will also treat valid values such as `0` and `""` as if no value had been supplied.

In [10]:
def show_limit(limit=None):
    if limit is None:
        limit = 10
    print("Limit:", limit)

show_limit()
show_limit(0)
show_limit(5)

Limit: 10
Limit: 0
Limit: 5


### A common bug: using `or` for every default

The expression `value or default` returns `default` whenever `value` is falsy. That can be useful, but it is wrong when `0`, `False`, or `""` should be preserved.

In [11]:
requested_limit = 0

limit_with_or = requested_limit or 10
print("Using 'or':", limit_with_or)

limit_with_none_check = (
    10 if requested_limit is None else requested_limit
)
print("Checking None explicitly:", limit_with_none_check)

Using 'or': 10
Checking None explicitly: 0


## 8. `and` and `or` return operands

In Python, `and` and `or` do not always return a Boolean. They return one of their operands:

- `x or y` returns `x` if `x` is truthy; otherwise it returns `y`.
- `x and y` returns `x` if `x` is falsy; otherwise it returns `y`.

This behavior is often used for defaults and short-circuiting, but it can be surprising if you expect the result always to be `True` or `False`.

In [12]:
print("hello" or "fallback")
print("" or "fallback")
print(0 or 99)
print("hello" and "world")
print(0 and "world")

hello
fallback
99
world
0


## 9. Truthiness with user input

`input()` always returns a string. Pressing Enter without typing anything gives `""`, which is falsy. That is different from `None`, which `input()` does not return on its own.

To keep this notebook runnable from top to bottom without pausing for keyboard input, the next cell simulates two possible inputs. When writing your own interactive program, replace the sample assignment with `user_text = input("Type something: ")`.


In [13]:
# Simulate two possible input() results:
# - "" represents pressing Enter without typing
# - "hello" represents typing text
for user_text in ("", "hello"):
    if user_text:
        print("You typed:", repr(user_text))
    else:
        print("You submitted empty text.")


You submitted empty text.
You typed: 'hello'


## 10. `None` as a function return value

A function returns `None` when it reaches the end without a `return` statement, or when it uses `return` without an expression.

In [14]:
def greet(name):
    print(f"Hello, {name}!")

def return_nothing_explicitly():
    return

first_result = greet("Asha")
second_result = return_nothing_explicitly()

print("Result from greet:", first_result)
print("Result from explicit return:", second_result)

Hello, Asha!
Result from greet: None
Result from explicit return: None


## 11. Empty is not the same as missing

Imagine a search function. An empty list could mean “the search ran and found no results.” `None` could mean “no search has been performed yet.” The exact meaning depends on your program, but choosing distinct representations can prevent confusion.

In [15]:
search_results = []
search_not_run = None

if search_results is None:
    print("Search has not run.")
elif not search_results:
    print("Search ran, but found no results.")
else:
    print("Search results:", search_results)

if search_not_run is None:
    print("The second search has not run yet.")

Search ran, but found no results.
The second search has not run yet.


## 12. Common mistakes

1. **Using `if not value` to check only for `None`.** This also catches zero, `False`, empty text, and empty collections.
2. **Using `value or default` when zero is a valid value.** It replaces zero with the default.
3. **Writing `value is False` to check general falsiness.** It only checks for the actual Boolean `False`; use `if not value` when you mean any falsy value.
4. **Assuming a function that prints something returns that printed result.** `print()` returns `None`.
5. **Using `== None` instead of `is None`.** Prefer identity checks for `None`.

In [16]:
value = 0

print("Is exactly None?", value is None)
print("Is exactly False?", value is False)
print("Is falsy?", not value)

Is exactly None? False
Is exactly False? False
Is falsy? True


## 13. Practice exercises

Try each task before opening the solution cells.

1. Create a variable called `middle_name` set to `None`. Print a message if it is missing.
2. Write `describe_input(value)` that returns `"missing"` for `None`, `"empty"` for an empty string, and `"provided"` for other values.
3. Given `quantity = 0`, choose a default of `5` only when `quantity is None`.
4. Print the truth value of `None`, `False`, `0`, `""`, `[]`, and `[0]`.
5. Explain in your own words why `value or 10` may be wrong for a numeric setting.

In [17]:
# Exercise 1
middle_name = None

if middle_name is None:
    print("Middle name is missing.")

Middle name is missing.


In [18]:
# Exercise 2
def describe_input(value):
    if value is None:
        return "missing"
    if value == "":
        return "empty"
    return "provided"

print(describe_input(None))
print(describe_input(""))
print(describe_input("Riya"))
print(describe_input(0))

missing
empty
provided
provided


In [19]:
# Exercise 3
quantity = 0
effective_quantity = 5 if quantity is None else quantity
print(effective_quantity)

0


In [20]:
# Exercise 4
values = [None, False, 0, "", [], [0]]

for value in values:
    print(f"{value!r:8} -> {bool(value)}")

None     -> False
False    -> False
0        -> False
''       -> False
[]       -> False
[0]      -> True


## 14. Review questions

Answer these without running code:

1. What does `None` represent?
2. What is the difference between `if value is None` and `if not value`?
3. Name four falsy values other than `None`.
4. Why can `value or 10` produce a bug?
5. What does a Python function return if it has no explicit `return` statement?
6. Does a truthy value have to be the Boolean `True`?
7. What is the difference between an empty list and `None` in a program that uses them to represent search results?

**Key takeaway:** Use `is None` when you mean “missing or explicitly None.” Use truthiness when you mean “empty/zero/false-like versus non-empty/non-zero.” Those are different checks.